# 1 Data Collection and Preparation

In order to structure and prepare the data set it is important to be sure what questions the data set should answer in the end.
The main questions are:
1. where is demand?
2. when is demand?
3. which factors determine demand?
4. how good can demand be projected?
5. topic charging (when, how strong, only private charging hubs, public infrastructure, ...)

To answer especially the forth question it is important to also consider which spatial (census tracts / hexagons) or time-based (hourly, 4-hourly) distribution works the best.

Final consulting should include in which districts to enter market at first (strategic), how many cars should be available when and where (tactical) and how to charge and reposition vehicles (operational).

- *1.1 Overview on dataset*
- *1.2 Syntactical Cleaning*
- *1.3 Semantic Validation and Outlier Handling*
- *1.4 Missing Value Strategy*
- *1.5 External Data Enrichment*
- *1.6 Spatial and Temporal Aggregation*
- *1.7 Result: Starting Point for further analysis*

Section 1.1 first provides a general overview of the raw dataset, including its columns, missing values, and basic structure. At the end of Section 1.1, the logic behind the overall preparation structure is explained in more detail. This is useful because the later cleaning and aggregation steps should not be arbitrary, but directly follow from the business questions and the intended final structure of the modeling dataset.


## 1.1 Overview on data set

To have a first glance into the data, some initial information about data types of each column and missing values per column should be considered, to maybe alright shorten the working set before going deeper into data preparation.

In [21]:
# the whole dataset until 4/30/26 is loaded and locally imported into the data folder, but it gets ignored by gitignore
# at first, only use the first 100.000 entries (of the total 50 million) to reduce computational effort
import pandas as pd

df = pd.read_csv(
    "../data/Taxi_Trips_(2024-)_20260502.csv",
    nrows=100_000
)

df_full = pd.read_csv(
    "../data/Taxi_Trips_(2024-)_20260502.csv"
)

df_full.head()

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,6c54cdb22905b181c23527e74c2b653a503107b7,db757f6c1157d9f81e266396132cd641837c189b803c52...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.407,"14,35",NaN,NaN,76.0,22.0,...,"$5,00","$48,01",Credit Card,Flash Cab,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,92276062","-87,699155343",POINT (-87.6991553432 41.9227606205)
1,6cd5d74b7ec8d7d230387900d72d74326dfb31de,f509c57d2f0b196f54d9b751ce2a1b6e956f378841b1e8...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.701,"19,14",1.703198e+10,1.703132e+10,76.0,32.0,...,"$0,00","$60,50",Credit Card,City Service,"41,97907082","-87,903039661",POINT (-87.9030396611 41.9790708201),"41,884987192","-87,620992913",POINT (-87.6209929134 41.8849871918)
2,d36a90d961ed60add20ab3a4a7a8ca98bd2bf929,4136627ef25b9fad79910c55679c02d8e1f2a42925d29c...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.260,"10,2",NaN,NaN,76.0,14.0,...,"$4,00","$31,25",Cash,Transit Administrative Center Inc,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,968069","-87,721559063",POINT (-87.7215590627 41.968069)
3,cb1fed57e7eeea1db20758256b754201f5e13e07,c9bda81f1aaad786527911c983b6be11f28c6f46469e1a...,04/01/2026 12:00:00 AM,04/01/2026 12:30:00 AM,1.608,"18,4",NaN,NaN,76.0,32.0,...,"$5,00","$50,50",Credit Card,Taxicab Insurance Agency Llc,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841)
4,d3c33c166fdbca45d040772d18bd1ae1dc669ade,f71223a469d78a2f65a090adc9d4fb5ae08dfc6694c650...,04/01/2026 12:00:00 AM,04/01/2026 12:00:00 AM,422.000,"0,98",NaN,NaN,32.0,28.0,...,"$0,00","$6,00",Cash,Chicago Independents,"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841),"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383)


In [22]:
# just to take a look at the data types it might be beneficial to print df.info()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 23 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Trip ID                     100000 non-null  str    
 1   Taxi ID                     100000 non-null  str    
 2   Trip Start Timestamp        100000 non-null  str    
 3   Trip End Timestamp          99998 non-null   str    
 4   Trip Seconds                99988 non-null   float64
 5   Trip Miles                  100000 non-null  str    
 6   Pickup Census Tract         45638 non-null   float64
 7   Dropoff Census Tract        44444 non-null   float64
 8   Pickup Community Area       97473 non-null   float64
 9   Dropoff Community Area      91399 non-null   float64
 10  Fare                        99892 non-null   str    
 11  Tips                        99892 non-null   str    
 12  Tolls                       99892 non-null   str    
 13  Extras                    

In [23]:
# in df.info() we see some missing values
# we test that based on the real data set to see if those structures really influence the whole data set
missing_overview = pd.DataFrame({
    "missing_count": df_full.isna().sum(),
    "missing_percent": df_full.isna().mean() * 100
})

missing_overview = missing_overview.sort_values(
    by="missing_count",
    ascending=False
)

missing_overview

,missing_count,missing_percent
Dropoff Census Tract,8435004,56.982301
Pickup Census Tract,8237431,55.647605
Dropoff Community Area,1320108,8.917932
Dropoff Centroid Location,1241359,8.385947
Dropoff Centroid Longitude,1241359,8.385947
Dropoff Centroid Latitude,1241359,8.385947
Pickup Community Area,413385,2.792604
Pickup Centroid Location,405819,2.741493
Pickup Centroid Longitude,405819,2.741493
Pickup Centroid Latitude,405819,2.741493


After assessing missing values and reviewing the available column information, the dataset structure is summarized to provide a basis for deciding which columns should be retained or excluded for the initial analysis.

| Column | Meaning | Relevance | Missing Values (%) |
|---|---|---|---:|
| `Trip ID` | Unique identifier of each trip | Technical key, e.g. for detecting duplicates | 0.00 |
| `Taxi ID` | Anonymized identifier of the taxi | Vehicle-level analysis, idle time, vehicle utilization | 0.00 |
| `Trip Start Timestamp` | Start time of the trip, rounded to 15-minute intervals | Demand analysis by time, weekday, and calendar period | 0.00 |
| `Trip End Timestamp` | End time of the trip, rounded to 15-minute intervals | Trip duration validation, temporal pattern analysis | 0.00 |
| `Trip Seconds` | Duration of the trip in seconds | Travel time, speed calculation, outlier detection | 0.02 |
| `Trip Miles` | Distance of the trip in miles | Trip length, efficiency, cost-related analysis | 0.00 |
| `Pickup Census Tract` | Census tract where the trip starts | Fine-grained spatial demand analysis | 55.65 |
| `Dropoff Census Tract` | Census tract where the trip ends | Destination areas, mobility flows | 56.98 |
| `Pickup Community Area` | Community area of the pickup location | Coarser spatial demand analysis | 2.79 |
| `Dropoff Community Area` | Community area of the dropoff location | Destination patterns, origin-destination analysis | 8.92 |
| `Fare` | Base fare of the trip | Revenue analysis, price level | 0.21 |
| `Tips` | Tip amount | Payment and service behavior; less central for demand prediction | 0.21 |
| `Tolls` | Toll costs | Cost component, rather a control variable | 0.21 |
| `Extras` | Additional charges | Additional costs, e.g. airport-related fees | 0.21 |
| `Trip Total` | Total amount paid for the trip | Revenue per trip | 0.21 |
| `Payment Type` | Payment method, e.g. cash or credit card | User behavior, data quality checks | 0.00 |
| `Company` | Taxi company operating the trip | Provider structure, possible filtering variable | 0.00 |
| `Pickup Centroid Latitude` | Latitude of the pickup area centroid | Mapping, H3 hexagons, spatial modeling | 2.74 |
| `Pickup Centroid Longitude` | Longitude of the pickup area centroid | Mapping, H3 hexagons, spatial modeling | 2.74 |
| `Pickup Centroid Location` | Point coordinate of the pickup area centroid | GIS processing; redundant if latitude and longitude are used | 2.74 |
| `Dropoff Centroid Latitude` | Latitude of the dropoff area centroid | Destination hotspots, spatial destination analysis | 8.39 |
| `Dropoff Centroid Longitude` | Longitude of the dropoff area centroid | Destination hotspots, spatial destination analysis | 8.39 |
| `Dropoff Centroid Location` | Point coordinate of the dropoff area centroid | GIS processing; redundant if latitude and longitude are used | 8.39 |

### Conceptual Considerations for Dataset Preparation

The raw taxi trip dataset consists of individual trip records. However, not every column is equally relevant for answering the business questions of the project. The final objective is not only to describe individual taxi trips, but to support a future ride-hailing fleet operator in understanding **where**, **when**, and under which conditions taxi demand occurs.

Therefore, the main analytical goal is to transform the transactional trip-level data into aggregated demand datasets. These datasets should allow us to answer questions such as:

- In which areas does taxi demand occur most frequently?
- At which times of day or days of the week is demand highest?
- How does demand vary across different spatial and temporal resolutions?
- Which resolution is most useful for operational fleet planning?
- How can external factors, such as weather, improve the understanding and prediction of demand?

The central target variable for the analysis is therefore:

**Demand count = number of taxi pickups per location unit and time bucket**

For example:

| Time Bucket | Location Unit | Demand Count |
|---|---|---:|
| 2024-01-01 08:00 | Community Area 32 | 57 |

This structure is much more relevant for the business problem than analyzing every taxi trip in isolation, because it directly reflects the amount of expected demand in a specific area at a specific time.

---

### Spatial and Temporal Aggregation Logic

A key question is how the final analytical dataset should be structured. Since the assignment requires the analysis of different spatial and temporal resolutions, several combinations should be considered.

#### Temporal resolution

Potential time buckets include:

- 15-minute intervals
- hourly intervals
- 4-hour intervals

A finer temporal resolution, such as 15 minutes, provides more detailed operational insights but may also lead to more volatile demand values. A coarser resolution, such as 4 hours, is more stable but less precise for short-term operational decisions.

#### Spatial resolution

Potential spatial units include:

- Census Tracts
- Community Areas
- H3 hexagons based on centroid coordinates

Community Areas offer a robust and interpretable spatial level with relatively few missing values. Census Tracts provide a more detailed spatial resolution but contain a high share of missing values. H3 hexagons allow for a flexible and standardized spatial grid, which can be adjusted depending on the required level of detail.

Combining three temporal and three spatial resolutions could theoretically lead to **nine aggregated datasets**:

| Spatial Resolution | Temporal Resolution |
|---|---|
| Community Area | 15 minutes |
| Community Area | 1 hour |
| Community Area | 4 hours |
| Census Tract | 15 minutes |
| Census Tract | 1 hour |
| Census Tract | 4 hours |
| H3 Hexagons | 15 minutes |
| H3 Hexagons | 1 hour |
| H3 Hexagons | 4 hours |

However, it may not be efficient to start with all nine datasets immediately. A more practical approach is to begin with one robust baseline dataset, for example **Community Area × 1 hour**, and then extend the analysis step by step.

---

### Handling Missing Spatial Data

Missing values should not be removed blindly. This is especially important for spatial variables such as Census Tracts. If more than half of the Census Tract values are missing, simply dropping these rows would strongly reduce the observed demand count and potentially distort the analysis.

For example, if only trips with available Census Tract information are retained, the resulting demand counts no longer represent total taxi demand. Instead, they only represent the subset of trips for which Census Tract information is available.

Therefore, the treatment of missing values should depend on the selected spatial resolution:

- For **Community Area-based datasets**, only rows with missing pickup community area need to be excluded.
- For **Census Tract-based datasets**, only rows with available pickup census tract can be used, but the limited data coverage must be clearly documented.
- For **H3-based datasets**, rows with missing pickup latitude or longitude need to be excluded.

A possible adjustment would be to scale observed Census Tract counts upward based on the share of available Census Tract values. However, this would assume that missing Census Tract values are randomly distributed. Since this assumption is likely uncertain, such an adjustment should be treated cautiously and not used as the main approach.

A more robust strategy is to use Community Areas and H3 hexagons as the main spatial levels and treat Census Tracts mainly as a sensitivity analysis.

---

### Transactional Data vs. Aggregated Data

It is important to distinguish between the raw transactional dataset and the final aggregated modeling dataset.

#### Transactional dataset

The transactional dataset contains one row per taxi trip. It is useful for:

- data quality checks
- duplicate detection
- outlier detection
- calculating trip durations and distances
- analyzing idle times between trips using the anonymized taxi ID
- calculating additional aggregated features later

Typical columns in this dataset include:

- trip start and end timestamp
- taxi ID
- trip seconds
- trip miles
- pickup and dropoff location information
- fare and trip total
- company and payment type

This dataset should be kept as a cleaned trip-level version because later analyses may require going back to the individual trip level.

#### Aggregated modeling dataset

The aggregated dataset contains one row per combination of time bucket and location unit. It is the main basis for descriptive demand analysis and predictive modeling.

A typical structure is:

| Time Bucket | Location Unit | Demand Count | Average Trip Miles | Average Trip Seconds | Average Trip Total | Weather Features |
|---|---|---:|---:|---:|---:|---|

Columns such as `Company`, `Payment Type`, `Trip ID`, or `Extras` are not directly useful in the aggregated dataset unless they are transformed into meaningful aggregate features, such as the number of different companies active in an area or the average fare per trip.

The aggregated dataset does not need to remain directly connected to individual `Trip ID`s. If additional features are needed later, they can be recalculated from the cleaned transactional dataset.

---

### Column Selection for the Initial Analysis

For the first analysis, the focus should be on columns that help answer the core demand question:

**How many taxi pickups occur in a specific area at a specific time?**

Therefore, the initial dataset should mainly include variables related to time, pickup location, basic trip characteristics, and possibly revenue.

The most important columns are:

- `Trip Start Timestamp`
- `Pickup Community Area`
- `Pickup Census Tract`
- `Pickup Centroid Latitude`
- `Pickup Centroid Longitude`
- `Trip Seconds`
- `Trip Miles`
- `Trip Total`

These columns are sufficient to create the first demand datasets and to describe demand patterns across time and space.

Some columns should be kept in the cleaned trip-level dataset, but do not need to be part of the first aggregated modeling dataset. This includes:

- `Taxi ID`, because it may be useful for idle time and vehicle utilization.
- `Trip End Timestamp`, because it is needed to validate trip duration and calculate idle time.
- `Dropoff Community Area`, `Dropoff Census Tract`, and dropoff coordinates, because they are useful for later origin-destination analysis.

Other columns are less relevant for the first demand model:

- `Trip ID`
- `Payment Type`
- `Company`
- `Fare`
- `Tips`
- `Tolls`
- `Extras`
- `Pickup Centroid Location`
- `Dropoff Centroid Location`

These columns should not necessarily be deleted from the raw data. However, they are not required for the first demand aggregation. Especially `Pickup Centroid Location` and `Dropoff Centroid Location` are redundant if latitude and longitude are already available.

---


Based on the information known to this point in time, a suitable structure to work on task 1 can be created. This structure includes the following points.

### 1.2 Syntactical Cleaning

This step focuses on technical issues in the data format.

The goal is to make the dataset technically usable before interpreting the values.

---

### 1.3 Missing Value Strategy

Before checking the semantic plausibility of individual values, missing values should first be analyzed. This is important because the later demand dataset is created by aggregating trips across spatial and temporal units.

Missing values should not be handled globally. Instead, the required columns depend on the intended spatial resolution.

For pickup-based demand, rows need a valid pickup time and a valid pickup location at the selected spatial level. Dropoff information is useful for later OD-analysis or repositioning, but it is not required for counting pickup demand.

The strategy should therefore be specific to each spatial resolution:

| Dataset Type | Required Location Information |
|---|---|
| Community Area dataset | `Pickup Community Area` |
| Census Tract dataset | `Pickup Census Tract` |
| H3 dataset | `Pickup Centroid Latitude` and `Pickup Centroid Longitude` |

This means that the number of usable rows differs across datasets. These differences should be documented clearly, especially for Census Tracts, where missing values are substantial.

---

### 1.4 Semantic Validation and Outlier Handling

After the missing value structure has been analyzed, the available values should be checked for plausibility.

Not every unusual value should be removed automatically. Some values may be rare but valid. For example, a short trip with a high price may look suspicious, but could still be possible due to congestion, waiting time, or additional charges.

Therefore, outliers should first be inspected and then either removed, capped, or flagged depending on the case. For the first demand analysis, the most important requirement is that the pickup time and the selected pickup location are valid.

---

### 1.5 External Data Enrichment

Weather data should be added after the time buckets have been created.

For hourly demand datasets, hourly weather data can be merged directly by timestamp. For 4-hour buckets, weather variables should be aggregated accordingly. 

Weather data should not be treated as a separate analysis only. It should become part of the final demand dataset, because weather may help explain why demand changes across time.

Furthermore other external data sources should be considered at this point.

---

### 1.6 Spatial and Temporal Aggregation

After cleaning and feature creation, the trip-level data can be aggregated.

The basic aggregation is:

**Group by time bucket and location unit, then count the number of pickups.**

This creates the main target variable:

`demand_count`

Additional aggregate variables can also be calculated, such as:

- average trip duration
- average trip distance
- average trip total
- number of unique taxis
- number of dropoffs
- average idle time, if calculated beforehand

---

### 1.7 Result: Starting Point for further analysis

The starting point is:

**Community Area × 1 hour**

This dataset is a suitable baseline because it has relatively few missing values, is easy to explain, and directly supports the main business question.

After this baseline is created, further versions can be tested e.g. on:

- Community Area × 4 hours
- H3 × 1 hour
- H3 × 4 hours
- Census Tract datasets as sensitivity checks

## 1.2 Syntactical Cleaning

This chapter focuses on technical data cleaning steps that are necessary before the dataset can be analyzed. The goal is to ensure that all columns have consistent formats and can be processed correctly.

The main steps include:

- standardizing column names
- converting timestamps into datetime format
- converting numeric and monetary columns into proper numeric types
- checking for duplicate rows

This step does not yet evaluate whether values are realistic in a business sense. It only ensures that the dataset is technically clean and usable.

For easier handling, the column names get standardized.

In [24]:
# Standardize column names for easier handling

df_full.columns = (
    df_full.columns
    .str.strip()                  # remove leading/trailing spaces
    .str.lower()                  # convert to lowercase
    .str.replace(" ", "_")         # replace spaces with underscores
)

df_full.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid__location'],
      dtype='str')

The trip start and end timestamps are currently stored as raw values. Converting them into datetime format is necessary to extract time-based features such as hour, weekday, month, and different time buckets for later demand aggregation.

In [25]:
timestamp_cols = [
    "trip_start_timestamp",
    "trip_end_timestamp"
]

for col in timestamp_cols:
    df_full[col] = pd.to_datetime(df_full[col], errors="coerce")

df_full[timestamp_cols].dtypes

/var/folders/6z/7sf2p5q96rj_nmwr3gxtj_tw0000gn/T/ipykernel_14783/1246117402.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_full[col] = pd.to_datetime(df_full[col], errors="coerce")
/var/folders/6z/7sf2p5q96rj_nmwr3gxtj_tw0000gn/T/ipykernel_14783/1246117402.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_full[col] = pd.to_datetime(df_full[col], errors="coerce")


trip_start_timestamp    datetime64[us]
trip_end_timestamp      datetime64[us]
dtype: object

Several columns contain numeric values such as trip duration, trip distance, coordinates, and trip costs. These columns need to be stored as numeric data types to enable filtering, aggregation, outlier detection, and later modeling. Even if the values appear numeric in the raw CSV file, they may initially be imported as strings and should therefore be converted explicitly.

In [27]:
# Convert trip_seconds: "1.407" -> 1407
df_full["trip_seconds"] = (
    df_full["trip_seconds"]
    .astype("string")
    .str.replace(".", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Convert decimal / monetary columns: "14,35" -> 14.35 and "$48,01" -> 48.01
num_cols = [
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "pickup_centroid_latitude",
    "pickup_centroid_longitude",
    "dropoff_centroid_latitude",
    "dropoff_centroid_longitude"
]

for col in num_cols:
    df_full[col] = (
        df_full[col]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

# Treat IDs / categories as strings
id_cols = [
    "trip_id",
    "taxi_id",
    "pickup_census_tract",
    "dropoff_census_tract",
    "pickup_community_area",
    "dropoff_community_area",
    "payment_type",
    "company"
]

for col in id_cols:
    df_full[col] = df_full[col].astype("string")

# Check result
df_full.dtypes

trip_id                               string
taxi_id                               string
trip_start_timestamp          datetime64[us]
trip_end_timestamp            datetime64[us]
trip_seconds                           Int64
trip_miles                           Float64
pickup_census_tract                   string
dropoff_census_tract                  string
pickup_community_area                 string
dropoff_community_area                string
fare                                 Float64
tips                                 Float64
tolls                                Float64
extras                               Float64
trip_total                           Float64
payment_type                          string
company                               string
pickup_centroid_latitude             Float64
pickup_centroid_longitude            Float64
pickup_centroid_location                 str
dropoff_centroid_latitude            Float64
dropoff_centroid_longitude           Float64
dropoff_ce

Types seem correct.

Duplicate records can distort demand counts because the same trip would be counted more than once. Therefore, the dataset is checked for fully duplicated rows and, additionally, for duplicated trip IDs.

In [28]:
# Check fully duplicated rows
n_full_duplicates = df_full.duplicated().sum()

print(f"Number of fully duplicated rows: {n_full_duplicates}")

Number of fully duplicated rows: 0


In [29]:
# Check duplicated Trip IDs
n_duplicate_trip_ids = df_full["trip_id"].duplicated().sum()

print(f"Number of duplicated trip IDs: {n_duplicate_trip_ids}")

Number of duplicated trip IDs: 0


After those initial checks, the data set is syntactical clean for the upcoming purposes.

## 1.3 Missing Value Strategy

After the dataset has been technically cleaned, the next step is to analyze missing values. This is especially important because the later demand dataset is created by aggregating trips across spatial and temporal units. Therefore, missing values must not be handled globally, but depending on the intended aggregation level.

In this dataset, missing values are not equally distributed across all spatial columns. Census Tract information has a very high share of missing values, while Community Area and centroid coordinates are much more complete. Therefore, dropping all rows with missing values would remove a large part of the dataset and could strongly distort the observed demand patterns.

The main checks in this section include:

- copy the number and percentage of missing values per column from 1.1
- comparing missing values across spatial columns:
  - `pickup_census_tract`
  - `pickup_community_area`
  - `pickup_centroid_latitude`
  - `pickup_centroid_longitude`
- checking whether missing pickup information affects certain times, areas, companies, or trip characteristics more strongly
- deciding which spatial columns are reliable enough for the main analysis
- defining separate filtering rules for each later spatial dataset:
  - Community Area dataset requires `pickup_community_area`
  - Census Tract dataset requires `pickup_census_tract`
  - H3 / hexagon dataset requires `pickup_centroid_latitude` and `pickup_centroid_longitude`
- avoiding a global drop of all missing values
- documenting why Census Tracts are only used carefully, for example as a sensitivity analysis
- documenting why Community Area × time is used as the first practical aggregation level

The central principle is that missing values are handled based on the analytical purpose of each dataset. For the initial demand model, trips only need a valid pickup time and pickup location at the selected spatial resolution. Other missing values, such as dropoff information or payment details, should not automatically lead to row removal because they are not required for counting pickup demand.

The actual removal of rows with missing values is therefore postponed until the corresponding aggregation dataset is created.

**TODO:** Explain every step and code to have a complete 1.3

## 1.4 Semantic Validation and Outlier Handling

After analyzing missing values, the next step is to check whether the available values are meaningful from a business and domain perspective. While syntactical cleaning ensures that columns have the correct format, semantic validation checks whether the recorded trips are plausible taxi trips.

This step builds on the missing value strategy from the previous section. Semantic checks are only performed where the required values are available. Missing values are therefore not removed globally, but handled depending on the specific validation or later aggregation task.

This step is important because implausible values can distort demand aggregation and model training. For example, trips with negative duration, negative distance, unrealistic timestamps, or extreme trip totals may not represent valid demand observations.

The main checks in this section include:

- validating the chronological order of trip start and trip end timestamps, where trip end needs to be after trip start
- checking trip duration values, especially zero, negative, and extremely long trips
- checking trip distance values, especially zero, negative, and extremely long trips
- identifying suspicious combinations of distance and price, while avoiding automatic exclusion because short but expensive trips may still be possible due to congestion, waiting time, or additional charges
- checking monetary values such as `trip_total` for negative or implausibly high amounts
- comparing `trip_seconds` with the duration calculated from start and end timestamps
- checking whether pickup coordinates are located within a plausible geographic range for Chicago
- checking whether Census Tracts are consistently linked to Community Areas, since Census Tracts are smaller spatial units and inconsistent mappings may indicate data quality issues
- analyzing companies with very few entries or strongly deviating trip characteristics
- checking for unusual or inconsistent payment types
- documenting which records should be excluded from later demand aggregation and why

The goal is not to make the dataset artificially clean, but to separate technically valid and analytically useful demand observations from records that are likely erroneous or not suitable for the intended demand modeling task. Extreme values are therefore first inspected and documented before any exclusion decision is made.

**TODO:** Explain every step and code to have a complete 1.4

## 1.5 Feature Engineering and External Data Enrichment

After missing values and semantic plausibility have been analyzed, additional variables are prepared to make the trip data useful for demand prediction and later fleet-related decisions. The goal is not to create as many features as possible, but to create only features that are clearly connected to the business questions.

The most important features are temporal features derived from `trip_start_timestamp`. Demand is expected to differ by hour, weekday, weekend, month, and season. Therefore, features such as `hour`, `day_of_week`, `is_weekend`, `month`, and `date` can be created. These features directly support the question of when demand occurs.

Spatial information is mainly prepared rather than newly created. Community Area and Census Tract identifiers already exist in the raw dataset and can be used later for aggregation after missing value checks. Census Tracts must be used carefully because they contain many missing values. H3 or hexagon identifiers, however, need to be newly created from pickup centroid coordinates.

Trip duration and trip distance are not direct causes of pickup demand, but they are useful for operational interpretation. Areas with the same number of pickups may still require different fleet sizes if trips differ strongly in length or duration. Therefore, features like `avg_speed_mph` are relevant for later utilization, charging, and repositioning analysis.

Monetary variables such as `trip_total`, price per mile, or price per minute are less central for the initial demand prediction task because they describe the realized trip rather than the demand event itself. They should therefore not be treated as core predictors of demand. However, features such as `price_per_mile` or `price_per_minute` may be useful later for assessing revenue potential or market attractiveness.

External data should be added with expected link to demand. Weather (`temperature`, `precipitation`, `snowfall`) should be linked to transactional rides. Regarding re-positioning especially the location of charging hubs may be really interesting to know. 

The main steps in this section include:

- creating temporal features from `trip_start_timestamp` (`hour`, `day_of_week`, `is_weekend`, `month`, `date`)
- creating H3 / hexagon identifiers from pickup centroid coordinates (`pickup_h3`)
- calculating selected operational trip-level features (`avg_speed_mph`)
- monetary features only for later business interpretation (`price_per_mile`, `price_per_minute`)
- enriching the dataset with weather data (`temperature`, `precipitation`)
- enriching the dataset with charging data

For the initial demand model, the core features are time and pickup location. Distance, duration, dropoff, monetary, and external variables are mainly used for later interpretation, fleet planning, charging strategy, and economic assessment.

**TODO:** Explain every step and code to have a complete 1.5

## 1.6 Spatial and Temporal Aggregation

After creating the relevant features, the trip-level dataset is transformed into demand observations. This is the key step where individual taxi trips are converted into demand per spatial and temporal unit.

The main target variable is `demand_count`, which represents the number of pickups within a specific location unit and time bucket. Each row in the aggregated dataset therefore describes how much demand occurred in one area during one time interval.

The aggregation can be created at different temporal and spatial resolutions.

Temporal resolutions:

- 15-minute intervals
- 1-hour intervals
- 4-hour intervals

Spatial resolutions:

- Community Area
- Census Tract
- H3 / hexagon cells based on pickup centroid coordinates

In theory, this results in nine possible aggregation datasets. However, the practical starting point is the Community Area × 1 hour dataset, because Community Area information has relatively few missing values and hourly demand is detailed enough for modeling while still being stable enough for interpretation.

The main steps in this section include:

- selecting the required pickup location column for each spatial resolution
- selecting the required time bucket for each temporal resolution
- filtering only the rows needed for the specific aggregation dataset
- grouping trips by location unit and time bucket
- calculating `demand_count` as the number of pickups
- adding aggregated trip characteristics, such as:
  - average trip distance
  - average trip duration
  - average trip total
  - number of unique taxis
- checking whether the resulting demand dataset has missing time-location combinations
- deciding whether missing combinations should be filled with zero demand
- documenting how many trips are usable for each aggregation level

For the initial modeling dataset, the first aggregation will be:

`pickup_community_area × 1 hour`

This dataset provides a clear and robust first view of where and when demand occurs. More granular datasets, such as H3 × 1 hour or Census Tract × 1 hour, can be created later for sensitivity checks or more detailed spatial analysis.

**TODO:** Explain every step and code to have a complete 1.6